# Nigerian Sign Language — Raw Data Diagnostics

**Goal:** Trying to audit images and PASCAL VOC bounding boxes, then export the eight required diagnostic figures as **300-DPI PNGs** into `figures/`.

## 0. Dependencies and configuration
Run the installation cell only if packages are missing. Restart the kernel after installing.

In [ ]:
%pip install -q numpy pandas pillow matplotlib scikit-learn scikit-image

In [ ]:
from pathlib import Path
from IPython.display import display
from collections import defaultdict
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.filters import laplace
from skimage.transform import resize
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

DATASET_ROOT = Path(
    '/Users/candacetarirohunzwi/Documents/Academics/'
    'AI System Design/'
    'sign-to-speech-for-sign-language-understanding-a-case-study-of-nigerian-sign-language/'
    'dataset'
)
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
MAX_PCA_IMAGES = 650
MAX_IMPORTANCE_IMAGES = 1800

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

def save_figure(fig, filename):
    dest = FIGURES_DIR / filename
    fig.savefig(dest, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print('Saved:', dest.resolve())

print('Dataset:', DATASET_ROOT)
print('Dataset exists:', DATASET_ROOT.is_dir())
print('Figures directory:', FIGURES_DIR.resolve())

## 1. Index images, read annotations, and calculate raw-image features

Each image becomes one row in `df`. We distinguish a **matching XML file** from a **successfully extracted, valid bounding box**—this matters because your earlier results showed 4,815 XML files but only 4,814 images with extracted boxes.

The validity check confirms that coordinates are finite, ordered, and inside the image. It does **not** verify that the box actually encloses the correct sign.

In [ ]:
def read_image(path):
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert('RGB'))

def stats_image(path):
    rgb = read_image(path)
    small = resize(rgb, (128,128), preserve_range=True, anti_aliasing=True).astype(np.float32) / 255.
    gray = rgb2gray(small)
    return dict(brightness=float(gray.mean()), contrast=float(gray.std()),
                blur_score=float(laplace(gray).var()), edge_density=float((np.abs(laplace(gray))>0.045).mean()),
                red_mean=float(small[:,:,0].mean()),green_mean=float(small[:,:,1].mean()),blue_mean=float(small[:,:,2].mean()))

def load_dataset():
    image_dir = DATASET_ROOT / 'images'
    xml_dir = DATASET_ROOT / 'annotations' / 'PASCAL_VOC'
    if not image_dir.is_dir() or not xml_dir.is_dir():
        raise FileNotFoundError(f'Expected {image_dir} and {xml_dir}. Set DATASET_ROOT correctly.')
    images = sorted(p for p in image_dir.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'})
    xmls = list(xml_dir.rglob('*.xml'))
    xml_by_stem = defaultdict(list)
    for p in xmls: xml_by_stem[p.stem].append(p)
    rows=[]
    for p in images:
        row={'path':str(p),'filename':p.name,'sign':p.parent.name,'xml_present':False,
             'xml_parsable':False,'box_present':False,'box_valid':False,
             'width':np.nan,'height':np.nan,'xmin':np.nan,'ymin':np.nan,'xmax':np.nan,'ymax':np.nan,
             'label_present':bool(p.parent.name),'image_readable':False}
        try:
            with Image.open(p) as im: row['width'],row['height']=ImageOps.exif_transpose(im).size; im.verify()
            row['image_readable']=True
        except Exception: pass
        matching=xml_by_stem.get(p.stem,[])
        if len(matching)>1: warnings.warn(f'Multiple XMLs for {p.name}: using {matching[0]}')
        if matching:
            row['xml_present']=True
            try:
                root=ET.parse(matching[0]).getroot()
                row['xml_parsable']=True
                objs=root.findall('./object')
                row['label_present']=bool(objs and objs[0].findtext('name'))
                boxes=root.findall('./object/bndbox')
                row['box_present']=len(boxes)>0
                for b in boxes:
                    try:
                        coords=[float(b.findtext(k)) for k in ('xmin','ymin','xmax','ymax')]
                        x1,y1,x2,y2=coords
                        if (np.isfinite(coords).all() and row['image_readable'] and
                            0<=x1<x2<=row['width'] and 0<=y1<y2<=row['height']):
                            row.update(dict(zip(('xmin','ymin','xmax','ymax'),coords)))
                            row['box_valid']=True
                            break
                    except (TypeError,ValueError): continue
            except (ET.ParseError,OSError): pass
        if row['image_readable']:
            try: row.update(stats_image(p))
            except Exception: pass
        rows.append(row)
    df=pd.DataFrame(rows)
    if df.empty: raise ValueError('No images found under images/.')
    df['box_width']=(df['xmax']-df['xmin'])/df['width']
    df['box_height']=(df['ymax']-df['ymin'])/df['height']
    df['box_area']=df['box_width']*df['box_height']
    df['box_aspect_ratio']=df['box_width']/df['box_height']
    df['x_center']=(df['xmin']+df['xmax'])/(2*df['width'])
    df['y_center']=(df['ymin']+df['ymax'])/(2*df['height'])
    print(f"Images: {len(df):,} | Classes: {df.sign.nunique()} | XMLs: {len(xmls):,} | Valid extracted boxes: {df.box_valid.sum():,}")
    df.to_csv(FIGURES_DIR/'indexed_image_metadata.csv',index=False)
    return df


df = load_dataset()

## 2. Dataset summary and investigate any unextracted boxes

In [ ]:
total_images = len(df)
total_classes = df["sign"].nunique()
total_xml = len(list(
    (DATASET_ROOT / "annotations" / "PASCAL_VOC").rglob("*.xml")
))
n_valid = int(df["box_valid"].sum())
n_without_boxes = total_images - n_valid

labels = ["Total images", "XML annotations", "Valid bounding boxes"]
values = [total_images, total_xml, n_valid]
colors = ["#4285F4", "#9063CD", "#17A398"]

fig, ax = plt.subplots(figsize=(9, 4.5))

bars = ax.barh(labels, values, color=colors, height=0.5)
ax.invert_yaxis()
ax.set_title("NSL Dataset Inventory", fontsize=15, fontweight="bold")
ax.set_xlabel("Count")
ax.set_xlim(0, max(values) * 1.2)
ax.grid(axis="x", alpha=0.15)
ax.set_axisbelow(True)

for bar, value in zip(bars, values):
    ax.text(
        value + max(values) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontweight="bold"
    )

plt.tight_layout()
save_figure(fig, "extra_dataset_inventory.png")

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------
# 1. Dataset Statistics
# ------------------------------------------------
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

total_images = len(df)
total_classes = df["sign"].nunique()

xml_count = len(list(
    (DATASET_ROOT / "annotations" / "PASCAL_VOC").rglob("*.xml")
))

matching_xml = int(df["xml_present"].sum())
valid_count = int(df["box_valid"].sum())
invalid_count = total_images - valid_count

coverage = (
    100 * valid_count / total_images
    if total_images else 0
)

# ------------------------------------------------
# 2. Chart Data
# ------------------------------------------------
metrics = [
    "Total Images",
    "Sign Classes",
    "XML Annotation Files",
    "Images with Matching XML",
    "Images with Valid Boxes",
    "Images without Valid Boxes"
]

counts = [
    total_images,
    total_classes,
    xml_count,
    matching_xml,
    valid_count,
    invalid_count
]

colors = [
    "#456B8C",  # Total images
    "#7896AF",  # Sign classes
    "#7896AF",  # XML files
    "#5C9EAD",  # Matching XML
    "#2A9D8F",  # Valid boxes
    "#D97767"   # Invalid boxes
]

# ------------------------------------------------
# 3. Create Figure
# ------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 5.5))

y = np.arange(len(metrics))

bars = ax.barh(
    y,
    counts,
    color=colors,
    height=0.58,
    edgecolor="none"
)

# Add count labels
offset = max(counts) * 0.015 if max(counts) else 0.5

for bar, value in zip(bars, counts):
    ax.text(
        bar.get_width() + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,}",
        va="center",
        fontsize=10,
        color="#25364A",
        fontweight="medium"
    )

# ------------------------------------------------
# 4. Formatting
# ------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(metrics, fontsize=10)

ax.invert_yaxis()

ax.set_xlabel(
    "Count",
    fontsize=11,
    labelpad=10
)

ax.set_title(
    "NSL Dataset: Annotation Completeness",
    fontsize=16,
    fontweight="bold",
    loc="left",
    pad=22
)

ax.set_xlim(0, max(counts) * 1.14 if max(counts) else 1)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.18
)
ax.set_axisbelow(True)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="y", length=0, pad=10)
ax.tick_params(axis="x", length=0, labelsize=9)

# Annotation coverage note
fig.text(
    0.13, 0.015,
    f"Valid bounding-box coverage: {coverage:.2f}%",
    fontsize=11,
    color="#2A9D8F",
    fontweight="bold"
)

# ------------------------------------------------
# 5. Save Figure
# ------------------------------------------------
plt.tight_layout(rect=[0, 0.055, 1, 1])

fig.savefig(
    FIGURES_DIR / "annotation_completeness.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# ------------------------------------------------
# 6. Identify Problematic Annotations
# ------------------------------------------------
problem_rows = df.loc[
    ~df["box_valid"].astype(bool),
    [
        "path",
        "sign",
        "xml_present",
        "xml_parsable",
        "box_present",
        "box_valid"
    ]
]

print(f"Images needing annotation review: {len(problem_rows)}")
print(f"Valid-box coverage: {coverage:.2f}%")

display(problem_rows)


# Intra Class Variation
- Intra-class variation examines how the same sign is performed or appears differently across multiple images, potentially involving different signers.

In [ ]:
# show_sign_grid(
#     signs=("A", "accept"),
#     per_sign=8,
#     draw_annotations=False
# )

# Overall Class Dsitribution

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ------------------------------------------------
# 1. Setup
# ------------------------------------------------
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

class_counts = df["sign"].value_counts().sort_values()
n_classes = len(class_counts)
mean_count = class_counts.mean()

# Professional color palette
CORAL = "#D97767"   # Bottom 20
BLUE = "#6488A8"    # Remaining classes
TEAL = "#288C83"    # Top 20
GRAY = "#737D8C"

bottom_classes = set(class_counts.head(20).index)
top_classes = set(class_counts.tail(20).index)

colors = [
    CORAL if sign in bottom_classes
    else TEAL if sign in top_classes
    else BLUE
    for sign in class_counts.index
]

# ------------------------------------------------
# 2. Figure dimensions
# ------------------------------------------------
fig_height = max(12, n_classes * 0.19)

fig, ax = plt.subplots(
    figsize=(11, fig_height),
    facecolor="white"
)

# ------------------------------------------------
# 3. Horizontal bar chart
# ------------------------------------------------
ax.barh(
    np.arange(n_classes),
    class_counts.values,
    color=colors,
    height=0.72,
    edgecolor="none"
)

# Mean reference line
ax.axvline(
    mean_count,
    color=GRAY,
    linestyle=(0, (4, 3)),
    linewidth=1.4,
    zorder=3
)

# ------------------------------------------------
# 4. Labels and axes
# ------------------------------------------------
ax.set_yticks(np.arange(n_classes))
ax.set_yticklabels(
    class_counts.index,
    fontsize=7.5
)

ax.set_xlabel(
    "Number of Images",
    fontsize=11,
    labelpad=10
)

ax.set_ylabel("")

ax.set_title(
    "Nigerian Sign Language Dataset — Class Distribution",
    fontsize=16,
    fontweight="bold",
    loc="left",
    pad=24
)

# Add space after the largest bar
ax.set_xlim(0, class_counts.max() * 1.08)

# Subtle vertical gridlines
ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.6,
    alpha=0.18
)
ax.set_axisbelow(True)

# Remove unnecessary chart borders
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.spines["bottom"].set_color("#D5DCE3")

ax.tick_params(
    axis="y",
    length=0,
    pad=6
)

ax.tick_params(
    axis="x",
    labelsize=9,
    length=0
)

# ------------------------------------------------
# 5. Legend
# ------------------------------------------------
legend_items = [
    Patch(facecolor=CORAL, label="Bottom 20 classes"),
    Patch(facecolor=BLUE, label="Middle classes"),
    Patch(facecolor=TEAL, label="Top 20 classes"),
    Line2D(
        [0], [0],
        color=GRAY,
        linestyle="--",
        linewidth=1.5,
        label=f"Mean: {mean_count:.1f} images"
    )
]

ax.legend(
    handles=legend_items,
    loc="lower right",
    fontsize=9,
    frameon=False,
    ncol=2
)

# ------------------------------------------------
# 6. Export
# ------------------------------------------------
fig.subplots_adjust(
    left=0.27,
    right=0.97,
    top=0.965,
    bottom=0.04
)

fig.savefig(
    FIGURES_DIR / "class_distribution_all_137.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    FIGURES_DIR / "class_distribution_all_137.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# ------------------------------------------------
# 7. Numerical summary
# ------------------------------------------------
print(f"Total classes: {n_classes}")
print(f"Total images: {class_counts.sum():,}")
print(f"Mean images per class: {mean_count:.2f}")
print(f"Minimum images in a class: {class_counts.min()}")
print(f"Maximum images in a class: {class_counts.max()}")


In [ ]:
top_classes = df["sign"].value_counts().head(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.viridis(
    np.linspace(0.25, 0.85, len(top_classes))
)

bars = ax.barh(
    top_classes.index,
    top_classes.values,
    color=colors,
    height=0.7
)

ax.set_title(
    "Top 20 Most Represented NSL Signs",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Number of images")
ax.set_xlim(0, max(top_classes.max(), 1) * 1.2)
ax.grid(axis="x", alpha=0.12)
ax.set_axisbelow(True)

for bar, value in zip(bars, top_classes.values):
    ax.text(
        value + top_classes.max() * 0.02,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()
save_figure(fig, "extra_top20_signs.png")

In [ ]:
# ------------------------------------------------
# BOTTOM 20 LEAST REPRESENTED NSL SIGNS
# ------------------------------------------------

bottom_classes = df["sign"].value_counts().tail(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.plasma(
    np.linspace(0.25, 0.85, len(bottom_classes))
)

bars = ax.barh(
    bottom_classes.index,
    bottom_classes.values,
    color=colors,
    height=0.7
)

ax.set_title(
    "Bottom 20 Least Represented NSL Signs",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Number of images")
ax.set_xlim(0, max(bottom_classes.max(), 1) * 1.2)
ax.grid(axis="x", alpha=0.12)
ax.set_axisbelow(True)

for bar, value in zip(bars, bottom_classes.values):
    ax.text(
        value + bottom_classes.max() * 0.02,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()

save_figure(fig, "extra_bottom20_signs.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageEnhance, ImageFilter
from pathlib import Path
import xml.etree.ElementTree as ET

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
BLUE = "#457B9D"
TEAL = "#2A9D8F"
ORANGE = "#F4A261"
RED = "#E76F51"
PURPLE = "#8064A2"
NAVY = "#243B53"

def export_figure(fig, filename):
    fig.savefig(
        FIGURES_DIR / filename,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()

def load_rgb(path):
    with Image.open(path) as im:
        return im.convert("RGB").copy()

def get_boxes(row):
    """Return valid (xmin, ymin, xmax, ymax) boxes for a row."""
    result = []

    # Prefer the structured box list
    if "boxes" in row.index and isinstance(row["boxes"], (list, tuple)):
        for b in row["boxes"]:
            if isinstance(b, dict):
                coords = [b.get(k) for k in ("xmin", "ymin", "xmax", "ymax")]
            elif isinstance(b, (list, tuple)) and len(b) >= 4:
                coords = list(b[:4])
            else:
                continue

            try:
                x1, y1, x2, y2 = map(float, coords)
                if np.isfinite([x1, y1, x2, y2]).all() and x2 > x1 and y2 > y1:
                    result.append((x1, y1, x2, y2))
            except (TypeError, ValueError):
                pass

    # Fallback to individual columns
    if not result and all(
        k in row.index for k in ("xmin", "ymin", "xmax", "ymax")
    ):
        try:
            x1, y1, x2, y2 = [
                float(row[k]) for k in ("xmin", "ymin", "xmax", "ymax")
            ]
            if np.isfinite([x1, y1, x2, y2]).all() and x2 > x1 and y2 > y1:
                result.append((x1, y1, x2, y2))
        except (TypeError, ValueError):
            pass

    return result

# Image Dimensions and Aspect Ratios
This examines whether images have consistent dimensions and aspect ratios, and identify variations in image resolution.

In [ ]:
# Read actual dimensions
dimension_records = []

for _, row in df.iterrows():
    try:
        with Image.open(row["path"]) as img:
            w, h = img.size

        dimension_records.append({
            "width": w,
            "height": h,
            "aspect_ratio": w / h,
            "resolution": f"{w} × {h}"
        })
    except (OSError, ValueError):
        continue

dims = pd.DataFrame(dimension_records)

if dims.empty:
    raise ValueError("No readable images found.")

# A. Image dimensions
resolution_counts = dims["resolution"].value_counts().head(15).sort_values()

fig, ax = plt.subplots(figsize=(9, max(4, len(resolution_counts)*0.5)))

bars = ax.barh(
    resolution_counts.index,
    resolution_counts.values,
    color=BLUE,
    height=0.6
)

ax.set_title("NSL Image Dimension Distribution", fontsize=15, fontweight="bold")
ax.set_xlabel("Number of Images")
ax.grid(axis="x", alpha=0.15)
ax.set_axisbelow(True)

for bar, value in zip(bars, resolution_counts.values):
    ax.text(
        bar.get_width() + max(resolution_counts.max()*0.01, 0.5),
        bar.get_y() + bar.get_height()/2,
        f"{value:,}",
        va="center",
        fontsize=10
    )

ax.set_xlim(0, resolution_counts.max()*1.18)
plt.tight_layout()
export_figure(fig, "image_dimensions.png")

# B. Aspect ratios
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.hist(
    dims["aspect_ratio"],
    bins=min(30, max(1, dims["aspect_ratio"].nunique())),
    color=PURPLE,
    edgecolor="white"
)

ax.axvline(
    dims["aspect_ratio"].median(),
    color=RED,
    linestyle="--",
    linewidth=2,
    label=f"Median: {dims['aspect_ratio'].median():.3f}"
)

ax.set_title("NSL Image Aspect Ratios", fontsize=15, fontweight="bold")
ax.set_xlabel("Width / Height")
ax.set_ylabel("Number of Images")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.15)
ax.set_axisbelow(True)

plt.tight_layout()
export_figure(fig, "image_aspect_ratios.png")

print("Unique image dimensions:", dims["resolution"].nunique())
print("Most common:", dims["resolution"].mode().iloc[0])

# Bounding Box Analysis
Examines the sizes, shapes and positions of sign annotations

In [ ]:
bbox_records = []

for _, row in df.iterrows():
    try:
        with Image.open(row["path"]) as im:
            W, H = im.size
    except OSError:
        continue

    for x1, y1, x2, y2 in get_boxes(row):
        bw = x2 - x1
        bh = y2 - y1

        if W <= 0 or H <= 0:
            continue

        bbox_records.append({
            "sign": row["sign"],
            "area_pct": 100 * bw * bh / (W * H),
            "aspect_ratio": bw / bh,
            "x_center": (x1 + x2) / (2 * W),
            "y_center": (y1 + y2) / (2 * H),
            "width_norm": bw / W,
            "height_norm": bh / H
        })

bbox_df = pd.DataFrame(bbox_records)

if bbox_df.empty:
    raise ValueError("No usable bounding boxes were found.")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# A. Box area
axes[0].hist(
    bbox_df["area_pct"],
    bins=35,
    color=TEAL,
    edgecolor="white"
)
axes[0].set_title("A. Bounding Box Area")
axes[0].set_xlabel("Percentage of Image (%)")
axes[0].set_ylabel("Number of Boxes")

# B. Box aspect ratio
axes[1].hist(
    bbox_df["aspect_ratio"],
    bins=35,
    color=ORANGE,
    edgecolor="white"
)
axes[1].set_title("B. Bounding Box Aspect Ratio")
axes[1].set_xlabel("Width / Height")

# C. Normalized bounding-box centers
hb = axes[2].hist2d(
    bbox_df["x_center"],
    bbox_df["y_center"],
    bins=25,
    range=[[0, 1], [0, 1]],
    cmap="Blues",
    cmin=1
)
axes[2].invert_yaxis()
axes[2].set_aspect("equal")
axes[2].set_title("C. Spatial Distribution")
axes[2].set_xlabel("Normalized X Center")
axes[2].set_ylabel("Normalized Y Center")

fig.colorbar(hb[3], ax=axes[2], label="Number of Boxes")

for ax in axes[:2]:
    ax.grid(axis="y", alpha=0.15)
    ax.set_axisbelow(True)

fig.suptitle(
    "NSL Bounding Box Characteristics",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
export_figure(fig, "bounding_box_analysis.png")

print("Total extracted boxes:", len(bbox_df))
print("Median box area (%):", round(bbox_df["area_pct"].median(), 2))
print("Median box aspect ratio:", round(bbox_df["aspect_ratio"].median(), 2))

# Annotation Coverage and Quality
Evaluates annotation completeness and identify missing, empty, malformed, or even out-of-bounds noudning boxes

In [ ]:
# ------------------------------------------------
# ANNOTATION COVERAGE AND QUALITY
# ------------------------------------------------

xml_dir = DATASET_ROOT / "annotations" / "PASCAL_VOC"
xml_lookup = {p.stem: p for p in xml_dir.rglob("*.xml")}

records = []

for _, row in df.iterrows():
    image_path = Path(row["path"])
    xml_path = xml_lookup.get(image_path.stem)

    status = "Valid"
    reason = ""

    if xml_path is None:
        status = "Missing XML"
        reason = "No matching XML file"

    else:
        try:
            root = ET.parse(xml_path).getroot()
            objects = root.findall("object")

            with Image.open(image_path) as im:
                W, H = im.size

            if not objects:
                status = "Empty annotation"
                reason = "No objects found"

            else:
                boxes = root.findall(".//object/bndbox")

                if not boxes:
                    status = "Empty annotation"
                    reason = "No bounding boxes found"

                else:
                    for box in boxes:
                        try:
                            x1 = float(box.findtext("xmin"))
                            y1 = float(box.findtext("ymin"))
                            x2 = float(box.findtext("xmax"))
                            y2 = float(box.findtext("ymax"))

                            if not np.isfinite([x1, y1, x2, y2]).all():
                                raise ValueError("Non-finite coordinate")

                            if not (0 <= x1 < x2 <= W and
                                    0 <= y1 < y2 <= H):
                                status = "Invalid bounding box"
                                reason = "Invalid or out-of-bounds coordinates"
                                break

                        except (TypeError, ValueError):
                            status = "Invalid bounding box"
                            reason = "Missing or non-numeric coordinates"
                            break

        except ET.ParseError:
            status = "Malformed XML"
            reason = "XML parsing error"

        except OSError:
            status = "Unreadable image"
            reason = "Could not read image"

    records.append({
        "image": image_path.name,
        "sign": row["sign"],
        "status": status,
        "reason": reason
    })

quality_df = pd.DataFrame(records)

status_order = [
    "Valid",
    "Missing XML",
    "Empty annotation",
    "Invalid bounding box",
    "Malformed XML",
    "Unreadable image"
]

counts = quality_df["status"].value_counts().reindex(
    status_order, fill_value=0
)

colors = [
    TEAL, ORANGE, PURPLE, RED, "#B56576", NAVY
]

# ------------------------------------------------
# PLOT: SEPARATE VALID AND PROBLEMATIC ANNOTATIONS
# ------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 4.8),
    gridspec_kw={"width_ratios": [0.8, 1.5]}
)

valid_count = int(counts["Valid"])
total_count = len(quality_df)
coverage = 100 * valid_count / total_count if total_count else 0

axes[0].bar(
    ["Valid annotations"],
    [valid_count],
    color=TEAL,
    width=0.45
)

axes[0].set_ylim(0, max(valid_count * 1.2, 1))
axes[0].set_title("A. Valid Annotations", fontweight="bold")
axes[0].set_ylabel("Number of Images")

axes[0].text(
    0, valid_count + max(valid_count * 0.03, 0.1),
    f"{valid_count:,}",
    ha="center",
    fontweight="bold",
    fontsize=13
)

problem_counts = counts.iloc[1:]

bars = axes[1].barh(
    problem_counts.index,
    problem_counts.values,
    color=colors[1:],
    height=0.55
)

axes[1].invert_yaxis()
axes[1].set_title("B. Annotation Issues", fontweight="bold")
axes[1].set_xlabel("Number of Images")
axes[1].set_xlim(0, max(problem_counts.max() * 1.3, 1.5))

for bar, value in zip(bars, problem_counts.values):
    axes[1].text(
        bar.get_width() + max(problem_counts.max() * 0.03, 0.04),
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontweight="bold"
    )

for ax in axes:
    ax.grid(axis="y", alpha=0.12)
    ax.set_axisbelow(True)

fig.suptitle(
    f"NSL Annotation Quality | {coverage:.2f}% Valid",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
export_figure(fig, "annotation_coverage_quality.png")

print(quality_df["status"].value_counts())
print("\nProblematic images:")
print(
    quality_df[quality_df["status"] != "Valid"]
    .to_string(index=False)
)

# Preprocessing Transformation Grid
Comparing different preprocessing techniques on the same original image to assess their effects on signing information. 

In [ ]:
# ------------------------------------------------
# PREPROCESSING TRANSFORMATION GRID
# ------------------------------------------------

# Choose an image that has a usable annotation
eligible_rows = df[
    df.apply(lambda row: len(get_boxes(row)) > 0, axis=1)
]

if eligible_rows.empty:
    raise ValueError("No images with bounding boxes available.")

sample = eligible_rows.sample(1, random_state=SEED).iloc[0]

original = load_rgb(sample["path"])

# 1. Resize, displayed without distortion
resized = original.copy()
resized.thumbnail((320, 320))

# 2. Brightness adjustment
brightened = ImageEnhance.Brightness(original).enhance(1.3)

# 3. Contrast enhancement
contrasted = ImageEnhance.Contrast(original).enhance(1.4)

# 4. Sharpening
sharpened = original.filter(
    ImageFilter.UnsharpMask(
        radius=1.5,
        percent=130,
        threshold=3
    )
)

# 5. Padded sign-region crop
W, H = original.size
x1, y1, x2, y2 = get_boxes(sample)[0]

padding_x = 0.15 * (x2 - x1)
padding_y = 0.15 * (y2 - y1)

crop_box = (
    max(0, int(x1 - padding_x)),
    max(0, int(y1 - padding_y)),
    min(W, int(x2 + padding_x)),
    min(H, int(y2 + padding_y))
)

cropped = original.crop(crop_box)

transformations = [
    ("Original", original),
    ("Resized", resized),
    ("Brightness +30%", brightened),
    ("Contrast Enhanced", contrasted),
    ("Sharpened", sharpened),
    ("Padded Sign Crop", cropped)
]

fig, axes = plt.subplots(
    2, 3,
    figsize=(13, 8),
    squeeze=False
)

for ax, (title, image) in zip(axes.flat, transformations):
    ax.imshow(image)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.axis("off")

fig.suptitle(
    f"NSL Preprocessing Transformation Grid | Sign: {sample['sign']}",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
export_figure(fig, "preprocessing_transformation_grid.png")

# Representative Class Image Grid

Displaying representative images from different sign classes to examine visual diversity across the dataset (compares different sign classes)

In [ ]:
# ------------------------------------------------
# REPRESENTATIVE NSL CLASS IMAGE GRID
# ------------------------------------------------

N_CLASSES = 16
N_COLS = 4
SEED = 42

# Select distinct available sign classes
available_classes = sorted(df["sign"].dropna().unique())

rng = np.random.default_rng(SEED)

selected_classes = rng.choice(
    available_classes,
    size=min(N_CLASSES, len(available_classes)),
    replace=False
)

n_rows = int(np.ceil(len(selected_classes) / N_COLS))

fig, axes = plt.subplots(
    n_rows,
    N_COLS,
    figsize=(13, 3.5 * n_rows),
    squeeze=False
)

for ax in axes.flat:
    ax.axis("off")

for ax, sign in zip(axes.flat, selected_classes):

    class_subset = df[df["sign"] == sign]

    # Use a reproducible random example
    sample = class_subset.sample(
        1,
        random_state=SEED
    ).iloc[0]

    try:
        image = load_rgb(sample["path"])

        ax.imshow(image)
        ax.set_title(
            str(sign),
            fontsize=11,
            fontweight="bold",
            pad=8
        )

    except (OSError, ValueError):
        ax.text(
            0.5, 0.5,
            "Image unavailable",
            ha="center",
            va="center",
            transform=ax.transAxes
        )

fig.suptitle(
    "Representative Images Across NSL Sign Classes",
    fontsize=17,
    fontweight="bold",
    y=1.01
)

plt.tight_layout()
export_figure(fig, "representative_class_image_grid.png")

# 8. Model Selection and YOLO26 Baseline

**Supervisor-directed baseline:** original images, the original 137-class distribution, original backgrounds, no foreground cropping, and no targeted class balancing. Source annotations are verified before conversion; standard YOLO training transforms must be logged.

**Why YOLO26n?** First evaluate the lightweight pretrained detection model (2.4M fused parameters; published COCO mAP50–95 40.9 and T4 TensorRT 1.7 ms). These are **external benchmarks**, not NSL outcomes. YOLO26s (9.5M parameters, COCO 48.6 mAP50–95, 2.5 ms) is a possible later comparison if accuracy requires it. Source: https://docs.ultralytics.com/models/yolo26/

**Before training:** validate original XML files, reconcile class-name order with source YOLO label mapping, verify duplicate/signer leakage constraints, and freeze train/val/test partitions. The notebook exploratory analysis does not establish leakage-free splitting by itself.

## 8.1 Reproducible full-dataset pipeline

The command below calls the shared, tested source script rather than reimplementing conversion and splitting in several notebook cells. **This code is not executed automatically**: verify `DATASET_ROOT` in Section 0 and review logs before training.

The script produces `verification.json`, manifests, a class map, an exclusion log, and 300-DPI plots. It does not perform signer-disjoint or perceptual-duplicate validation, which requires further metadata/review.

In [ ]:
from pathlib import Path
import sys, subprocess
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "phase2_pipeline.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src" / "phase2_pipeline.py").exists(), "Run this notebook from the assignment project or its notebooks/ folder"
print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_ROOT)
# Uncomment after reviewing labels and split policy:
# subprocess.run([sys.executable, str(PROJECT_ROOT / "src" / "phase2_pipeline.py"),
#                 "--dataset", str(DATASET_ROOT), "--out", str(PROJECT_ROOT / "outputs" / "phase2"),
#                 "--export-yolo"], check=True)

## 8.2 Verify complete preprocessing output

Only report numbers read from the actual full-dataset `verification.json`. Check the exclusion log, per-split class coverage, SHA-256 overlap, mapping consistency, and possible signer/session overlap. Image tensors are letterboxed at model runtime; do not confuse the five-column audit-only feature matrix with detector inputs.

In [ ]:
import json
verification_file = PROJECT_ROOT / "outputs" / "phase2" / "verification.json"
if verification_file.exists():
    verification = json.loads(verification_file.read_text())
    for k,v in verification.items(): print(f"{k}: {v}")
else:
    print("Full-dataset pipeline has not been run at", verification_file)

## 8.3 YOLO26n baseline training (only after audit approval)

Run the following command in a terminal after verifying the split, XML conversion, and 137-class ID mapping. No resampling, segmentation, or oracle crops are applied. Default training augmentation is part of the model recipe and should be logged.

```bash
python src/train_baseline.py --data outputs/phase2/yolo_dataset/data.yaml --epochs 100 --batch 16
```

Validation output goes to `baseline_validation_metrics.json`. Keep the held-out test set untouched until model selection is finalized.

## 8.4 Proposed follow-up experiments (not Assignment 3 results)

1. Minority-class augmentation/oversampling, only after measuring per-class baseline AP.
2. Background-removed target-signer images.
3. Padded sign-region crops (oracle crops must not be compared as deployable full-image inference).

Vary one factor at a time and preserve original-image split membership.

**Evidence boundary:** the figures above are EDA; this notebook alone does not prove the YOLO26 model has trained.